# AKTU Autonomy Backend – Colab Runner (v2)

Run the **FastAPI backend** of the AKTU Academic Autonomy Portal from **Google Colab**.

**Steps:**
1. Run the first code cell to mount Drive, clone/pull the private GitHub repo, install backend dependencies, and run Alembic migrations.
2. Run the second code cell to start Uvicorn and open an **ngrok** tunnel. It will print a public URL and `/api/health` link.
3. Keep the notebook running while you use the API from your browser or frontend.

> You will be prompted for a **GitHub Personal Access Token (PAT)** with `repo` scope to clone/pull the private monorepo.


# AKTU Autonomy Backend – Colab Runner\n\nThis notebook installs dependencies, mounts Google Drive, starts the FastAPI backend with Uvicorn, exposes it via ngrok, and prints the public URL.\n

In [ ]:
# 1) Mount Drive, clone/pull repo, install deps, run migrations

from google.colab import drive
import getpass
import os
import subprocess

# Mount Google Drive for persistent DB + uploads
drive.mount("/content/drive")

# --- Configure repo location and GitHub URL ---
REPO_DIR = "/content/aktu-autonomy-portal"
# Replace this with your private repo URL if different
REPO_URL = os.environ.get("AKTU_REPO_URL", "https://github.com/ORG/aktu-autonomy-portal.git")

# --- Clone or pull ---
if not os.path.exists(REPO_DIR):
    print("Cloning repo…")
    token = getpass.getpass("GitHub PAT (repo scope): ")
    auth_url = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", auth_url, REPO_DIR], check=True)
else:
    print("Repo already exists; pulling latest changes…")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# --- Configure backend paths (DB and uploads on Drive) ---
os.environ.setdefault("AKTU_DB_PATH", "/content/drive/MyDrive/aktu_autonomy/aktu_autonomy.db")
os.environ.setdefault("UPLOAD_DIR", "/content/drive/MyDrive/aktu_autonomy/uploads")
os.makedirs(os.path.dirname(os.environ["AKTU_DB_PATH"]), exist_ok=True)
os.makedirs(os.environ["UPLOAD_DIR"], exist_ok=True)

# --- Install Python deps ---
subprocess.run(["pip", "install", "-q", "-r", "backend/requirements.txt", "pyngrok"], check=True)

# --- Run Alembic migrations ---
os.chdir(os.path.join(REPO_DIR, "backend"))
subprocess.run(["alembic", "upgrade", "head"], check=True)

print("Backend dependencies installed and migrations applied.")


In [ ]:
# 2) Start Uvicorn + ngrok tunnel

from pyngrok import ngrok
import threading
import uvicorn
import os

REPO_DIR = "/content/aktu-autonomy-portal"
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
os.chdir(BACKEND_DIR)

port = 8000
public_url = ngrok.connect(port, "http").public_url
print("Public URL:", public_url)
print("Health endpoint:", public_url + "/api/health")


def run_backend():
  # app.main:app, run inside backend directory
  uvicorn.run("app.main:app", host="0.0.0.0", port=port, reload=False)

thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

print("Backend is starting… you can now call the health URL above.")


In [ ]:
# (Optional) Inspect environment / debug

import os
print("PWD:", os.getcwd())
print("AKTU_DB_PATH:", os.environ.get("AKTU_DB_PATH"))
print("UPLOAD_DIR:", os.environ.get("UPLOAD_DIR"))

